In [ ]:
import uuid
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOGO = "desafio_vendas"
T_BRONZE     = f"{CATALOGO}.bronze.vendas_raw"
T_SILVER     = f"{CATALOGO}.silver.vendas"
T_QUARENTENA = f"{CATALOGO}.silver.vendas_quarentena"
T_QUALIDADE  = f"{CATALOGO}.ops.qualidade_resultados"
T_LOG        = f"{CATALOGO}.ops.log_pipeline"

ID_LOTE = str(uuid.uuid4())
INICIO = datetime.now()
SALT_PII = "poc-desafio-vendas"   # PoC. Em produção, ler de um secret scope.

In [ ]:
MAPA_PRODUTO = {"notebook": "Notebook", "not-book": "Notebook",
                "smartphone": "Smartphone", "smart phon": "Smartphone",
                "smartwatch": "Smartwatch", "smart watchi": "Smartwatch"}
TIPO_ESPERADO = {"notebook": "Informática", "smartphone": "Eletrônico", "smartwatch": "Acessório"}

MAPA_STATUS_ENVIO = {"em processamento": "Em processamento", "em trânsito": "Em trânsito",
                     "enviado": "Enviado", "entregue": "Entregue"}
MAPA_STATUS_VENDA = {"aprovada": "Aprovada", "cancelada": "Cancelada",
                     "pendente": "Pendente", "desconhecido": "Desconhecido"}
MAPA_SEXO = {"m": "M", "f": "F", "outro": "Outro", "1": "M", "2": "F"}  # 1=M, 2=F (ISO 5218): premissa documentada
MAPA_SIM_NAO = {"sim": True, "não": False}

CIDADES = {  # chave minúscula: (nome canônico, UF)
    "ananindeua": ("Ananindeua", "PA"), "belo horizonte": ("Belo Horizonte", "MG"),
    "belém": ("Belém", "PA"), "campinas": ("Campinas", "SP"), "caucaia": ("Caucaia", "CE"),
    "caxias do sul": ("Caxias do Sul", "RS"), "curitiba": ("Curitiba", "PR"),
    "feira de santana": ("Feira de Santana", "BA"), "florianópolis": ("Florianópolis", "SC"),
    "fortaleza": ("Fortaleza", "CE"), "joinville": ("Joinville", "SC"), "londrina": ("Londrina", "PR"),
    "niterói": ("Niterói", "RJ"), "olinda": ("Olinda", "PE"), "porto alegre": ("Porto Alegre", "RS"),
    "recife": ("Recife", "PE"), "rio de janeiro": ("Rio de Janeiro", "RJ"),
    "salvador": ("Salvador", "BA"), "são paulo": ("São Paulo", "SP"), "uberlândia": ("Uberlândia", "MG"),
}
UFS_VALIDAS = {"BA", "CE", "MG", "PA", "PE", "PR", "RJ", "RS", "SC", "SP"}

# Formatos por coluna (regex de guarda, formato Spark)
FORMATOS_VENDA = [(r"^\d{4}-\d{2}-\d{2}$", "yyyy-MM-dd"),
                  (r"^\d{2}/\d{2}/\d{4}$", "dd/MM/yyyy"),
                  (r"^\d{2}-\d{2}-\d{4}$", "MM-dd-yyyy")]
FORMATOS_RETIRADA = [(r"^\d{4}-\d{2}-\d{2}$", "yyyy-MM-dd"),
                     (r"^\d{4}/\d{2}/\d{2}$", "yyyy/MM/dd"),
                     (r"^\d{2}-\d{2}-\d{4}$", "dd-MM-yyyy")]

In [ ]:
def mapear(coluna, mapa, padrao=None):
    """Mapeia valores pela chave em minúsculas e sem espaços nas pontas. Sem correspondência = padrao."""
    chave = F.lower(F.trim(F.col(coluna)))
    expr = F.lit(padrao)
    for k, v in mapa.items():
        expr = F.when(chave == F.lit(k), F.lit(v)).otherwise(expr)
    return expr

def parse_data(coluna, regras):
    expr = F.lit(None).cast("date")
    for regex, fmt in reversed(regras):
        expr = F.when(F.col(coluna).rlike(regex),
                      F.expr(f"try_to_date({coluna}, '{fmt}')")).otherwise(expr)
    return expr

def to_int(c):         return F.expr(f"try_cast(try_cast({c} as double) as int)")
def to_dec(c, p, s):   return F.expr(f"try_cast({c} as decimal({p},{s}))")

In [ ]:
b = spark.table(T_BRONZE)
colunas_neg = [c for c in b.columns if not c.startswith("_")]

def limpa(c):  # trim e vazio vira NULL
    return F.when(F.trim(F.col(c)) == "", F.lit(None)).otherwise(F.trim(F.col(c))).alias(c)

t = b.select(*[limpa(c) for c in colunas_neg],
             "_numero_linha_origem", "_arquivo_origem", F.col("_id_lote").alias("_id_lote_bronze"))

cpf_dig  = F.regexp_replace("cliente_cpf", r"\D", "")
cpf_ok   = F.coalesce(F.length(cpf_dig) == 11, F.lit(False))
email_l  = F.lower("cliente_email")
email_ok = F.coalesce(email_l.rlike(r"^[^@\s]+@[^@\s]+\.[^@\s]+$"), F.lit(False))
tel_ok   = F.coalesce(F.col("cliente_telefone").rlike(r"^\(\d{2}\) \d{5}-\d{4}$"), F.lit(False))

uf_original = F.when(F.upper("estado").isin(*sorted(UFS_VALIDAS)), F.upper("estado"))
uf_cidade   = mapear("cidade", {k: v[1] for k, v in CIDADES.items()})

n = t.select(
    "_numero_linha_origem", "_arquivo_origem", "_id_lote_bronze",
    "cliente",
    mapear("produto", MAPA_PRODUTO).alias("produto"),
    "tipo_produto",
    parse_data("data_venda", FORMATOS_VENDA).alias("data_venda"),
    parse_data("data_retirada", FORMATOS_RETIRADA).alias("data_retirada"),
    to_dec("valor_frete", 10, 2).alias("valor_frete"),
    mapear("status_envio", MAPA_STATUS_ENVIO).alias("status_envio"),
    mapear("cidade", {k: v[0] for k, v in CIDADES.items()}).alias("cidade"),
    F.coalesce(uf_cidade, uf_original).alias("uf"),
    (uf_cidade.isNotNull() & (uf_original.isNull() | (uf_original != uf_cidade))).alias("f_uf_corrigida"),
    mapear("cliente_sexo", MAPA_SEXO).alias("cliente_sexo"),
    to_int("qtd_vendida").alias("qtd_vendida"),
    to_dec("valor_unitario", 12, 2).alias("valor_unitario"),
    mapear("status_venda", MAPA_STATUS_VENDA).alias("status_venda"),
    "tipo_pagamento",
    to_int("avaliacao_produto").alias("avaliacao_produto"),
    mapear("status_desconto", MAPA_SIM_NAO).alias("desconto_aplicado"),
    to_int("desconto_pct").alias("desconto_pct"),
    mapear("item_promocional", MAPA_SIM_NAO).alias("item_promocional"),
    # --- LGPD: a Silver não guarda CPF, telefone ou e-mail em claro ---
    F.when(cpf_ok, F.concat(F.lit("***.***.***-"), F.substring(cpf_dig, 10, 2))).alias("cliente_cpf_mascarado"),
    F.when(cpf_ok, F.sha2(F.concat(F.lit(SALT_PII), cpf_dig), 256)).alias("cliente_cpf_hash"),
    (cpf_ok & ~F.col("cliente_cpf").rlike(r"^\d{3}\.\d{3}\.\d{3}-\d{2}$")).alias("f_cpf_corrigido"),
    F.when(tel_ok, F.regexp_replace("cliente_telefone", r"^(\(\d{2}\)) \d{5}-(\d{4})$", "$1 *****-$2")).alias("cliente_telefone_mascarado"),
    F.when(email_ok, F.regexp_replace(email_l, r"^(.)[^@]*(@.*)$", "$1*****$2")).alias("cliente_email_mascarado"),
)

In [ ]:
tipo_esperado = mapear("produto", TIPO_ESPERADO)
ENVIO_ATIVO = ["Enviado", "Entregue", "Em trânsito"]

REGRAS = [  # (id, descrição, condição que indica o problema, severidade)
    ("R01", "data_venda ausente ou em formato não reconhecido", F.col("data_venda").isNull(), "ERRO"),
    ("R02", "produto fora do dicionário", F.col("produto").isNull(), "ERRO"),
    ("R03", "qtd_vendida nula ou <= 0", F.coalesce(F.col("qtd_vendida") <= 0, F.lit(True)), "ERRO"),
    ("R04", "valor_unitario nulo ou <= 0", F.coalesce(F.col("valor_unitario") <= 0, F.lit(True)), "ERRO"),
    ("R05", "desconto_pct nulo ou fora de 0..100", F.coalesce(~F.col("desconto_pct").between(0, 100), F.lit(True)), "ERRO"),
    ("R06", "data_retirada ausente ou não reconhecida", F.col("data_retirada").isNull(), "ALERTA"),
    ("R07", "retirada anterior à venda", F.col("data_retirada") < F.col("data_venda"), "ALERTA"),
    ("R08", "venda cancelada com envio em andamento ou concluído",
        (F.col("status_venda") == "Cancelada") & F.col("status_envio").isin(*ENVIO_ATIVO), "ALERTA"),
    ("R09", "status de desconto incoerente com o percentual",
        (~F.col("desconto_aplicado") & (F.col("desconto_pct") > 0)) | (F.col("desconto_aplicado") & (F.col("desconto_pct") == 0)), "ALERTA"),
    ("R10", "valor_frete ausente", F.col("valor_frete").isNull(), "ALERTA"),
    ("R11", "cidade ausente ou fora do dicionário", F.col("cidade").isNull(), "ALERTA"),
    ("R12", "UF indeterminada", F.col("uf").isNull(), "ALERTA"),
    ("R13", "UF do arquivo divergente da cidade (corrigida pela cidade)", F.col("f_uf_corrigida"), "ALERTA"),
    ("R14", "avaliacao_produto ausente", F.col("avaliacao_produto").isNull(), "ALERTA"),
    ("R15", "CPF ausente ou inválido", F.col("cliente_cpf_hash").isNull(), "ALERTA"),
    ("R16", "CPF corrigido (caracteres extras removidos)", F.col("f_cpf_corrigido"), "ALERTA"),
    ("R17", "e-mail ausente ou inválido", F.col("cliente_email_mascarado").isNull(), "ALERTA"),
    ("R18", "status_venda desconhecido", F.coalesce(F.col("status_venda") == "Desconhecido", F.lit(True)), "ALERTA"),
    ("R19", "tipo_produto incoerente com o produto", F.col("tipo_produto") != tipo_esperado, "ALERTA"),
]

def lista_regras(sev):
    itens = [F.when(F.coalesce(cond, F.lit(False)), F.lit(rid))
             for rid, _, cond, s in REGRAS if s == sev]
    return F.filter(F.array(*itens), lambda x: x.isNotNull())

In [ ]:
cols_dedup = ["produto", "data_venda", "data_retirada", "valor_frete", "status_envio", "cidade", "uf",
              "cliente", "cliente_cpf_hash", "cliente_sexo", "tipo_produto", "qtd_vendida", "valor_unitario",
              "status_venda", "tipo_pagamento", "avaliacao_produto", "desconto_aplicado", "desconto_pct",
              "item_promocional"]

w = Window.partitionBy(*cols_dedup).orderBy("_numero_linha_origem")
n = n.withColumn("_rn", F.row_number().over(w))
duplicadas = n.filter("_rn > 1")
unicas = n.filter("_rn = 1")

v = (unicas.withColumn("_erros", lista_regras("ERRO"))
           .withColumn("alertas", lista_regras("ALERTA")))
validas    = v.filter(F.size("_erros") == 0)
rejeitadas = v.filter(F.size("_erros") > 0)

# Quarentena: sem PII, só rastreio + campos brutos que explicam a rejeição
brutas = b.select("_numero_linha_origem", "cliente",
                  F.col("produto").alias("produto_bruto"),
                  F.col("data_venda").alias("data_venda_bruta"),
                  F.col("qtd_vendida").alias("qtd_vendida_bruta"),
                  F.col("valor_unitario").alias("valor_unitario_bruto"))

q_erro = rejeitadas.select("_numero_linha_origem", F.lit("ERRO_QUALIDADE").alias("tipo_rejeicao"), F.col("_erros").alias("regras"))
q_dup  = duplicadas.select("_numero_linha_origem", F.lit("DUPLICADA").alias("tipo_rejeicao"), F.array(F.lit("DUP")).alias("regras"))

quarentena = (q_erro.unionByName(q_dup)
              .join(brutas, "_numero_linha_origem", "left")
              .withColumn("_id_lote_silver", F.lit(ID_LOTE))
              .withColumn("_data_processamento", F.current_timestamp()))

In [ ]:
silver = (validas
    .withColumn("id_venda", F.row_number().over(Window.orderBy("_numero_linha_origem")))
    .select("id_venda", "produto", "tipo_produto", "data_venda", "data_retirada", "valor_frete",
            "status_envio", "cidade", "uf", "cliente", "cliente_cpf_mascarado", "cliente_cpf_hash",
            "cliente_sexo", "cliente_telefone_mascarado", "cliente_email_mascarado",
            "qtd_vendida", "valor_unitario", "status_venda", "tipo_pagamento", "avaliacao_produto",
            "desconto_aplicado", "desconto_pct", "item_promocional", "alertas",
            "_arquivo_origem", "_id_lote_bronze", "_numero_linha_origem",
            F.lit(ID_LOTE).alias("_id_lote_silver"),
            F.current_timestamp().alias("_data_processamento")))

(silver.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable(T_SILVER))
(quarentena.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable(T_QUARENTENA))

spark.sql(f"COMMENT ON TABLE {T_SILVER} IS 'Vendas tratadas e padronizadas. PII mascarada. Fonte: {T_BRONZE}'")
spark.sql(f"COMMENT ON TABLE {T_QUARENTENA} IS 'Registros da Bronze rejeitados na Silver (erro de qualidade ou duplicata), com a regra violada'")

In [ ]:
# Resultado por regra (sobre os registros únicos)
agg = v.agg(F.count("*").alias("_total"),
            *[F.sum(F.when(F.coalesce(c, F.lit(False)), 1).otherwise(0)).alias(rid)
              for rid, _, c, _ in REGRAS]).collect()[0].asDict()

linhas = [(ID_LOTE, "silver", rid, desc, sev, int(agg[rid]), int(agg["_total"]),
           round(100.0 * agg[rid] / agg["_total"], 2), datetime.now())
          for rid, desc, _, sev in REGRAS]
(spark.createDataFrame(linhas,
    "id_lote string, etapa string, regra_id string, descricao string, severidade string, "
    "qtd_afetada long, qtd_total long, pct_afetado double, data_execucao timestamp")
 .write.mode("append").saveAsTable(T_QUALIDADE))

# Reconciliação: bronze = silver + rejeitadas + duplicadas
qtd_bronze     = b.count()
qtd_dup_brutas = qtd_bronze - b.select(*colunas_neg).distinct().count()
qtd_dup_norm   = duplicadas.count()
qtd_rejeitada  = rejeitadas.count()
qtd_silver     = spark.table(T_SILVER).count()
ok = (qtd_bronze == qtd_silver + qtd_rejeitada + qtd_dup_norm)
print(f"Bronze={qtd_bronze} | Silver={qtd_silver} | Rejeitadas={qtd_rejeitada} | "
      f"Duplicadas (pós-normalização)={qtd_dup_norm} | Duplicatas exatas no bruto={qtd_dup_brutas} | Reconciliado={ok}")

log = spark.createDataFrame(
    [(ID_LOTE, "silver", T_SILVER, qtd_bronze, qtd_silver, "SUCESSO" if ok else "DIVERGENCIA",
      INICIO, datetime.now(), qtd_rejeitada, qtd_dup_norm)],
    "id_lote string, etapa string, tabela_destino string, qtd_lida long, qtd_gravada long, status string, "
    "inicio timestamp, fim timestamp, qtd_rejeitada long, qtd_duplicada long")
log.write.mode("append").option("mergeSchema", "true").saveAsTable(T_LOG)

In [ ]:
display(spark.table(T_QUALIDADE).filter(F.col("id_lote") == ID_LOTE).orderBy("regra_id"))
display(spark.table(T_QUARENTENA).groupBy("tipo_rejeicao", F.col("regras").cast("string").alias("regras")).count().orderBy(F.desc("count")))
spark.table(T_SILVER).printSchema()
display(spark.table(T_SILVER).limit(10))
display(spark.table(T_SILVER).agg(F.min("data_venda"), F.max("data_venda"), F.min("data_retirada"), F.max("data_retirada")))

In [ ]:
display(spark.table(T_BRONZE).select(
    (F.expr("try_cast(qtd_vendida as double)") < 0).alias("qtd_negativa"),
    (F.expr("try_cast(valor_unitario as double)") < 0).alias("preco_negativo"),
    F.trim("status_venda").alias("status_venda"))
  .groupBy("qtd_negativa", "preco_negativo", "status_venda").count()
  .orderBy("qtd_negativa", "preco_negativo", "status_venda"))

In [ ]:
import json
dbutils.notebook.exit(json.dumps({
    "id_lote": ID_LOTE, "qtd_lida": qtd_bronze, "qtd_gravada": qtd_silver,
    "qtd_rejeitada": qtd_rejeitada, "qtd_duplicada": qtd_dup_norm,
    "status": "SUCESSO" if ok else "DIVERGENCIA"}))